# 📄 Fase 4 — Salida: PDF + PostgreSQL
**Pipeline DIGI Spain Telecom — Control de Calidad C2C**

**Nodos:** 12 (PDF con ReportLab) · 13 (INSERT PostgreSQL)

**Entrada:** `outputs/resultado_fase3.json`  
**Salida:** `outputs/informe_auditoria.pdf` + registro en PostgreSQL

---

## ⚠️ PASO 0 — Auto-fix NumPy (ejecutar siempre primero)

In [ ]:
import subprocess, sys, importlib.util
spec = importlib.util.find_spec('numpy')
if spec:
    import numpy as _np
    version = tuple(int(x) for x in _np.__version__.split('.')[:2])
    if version >= (2, 0):
        print(f'⚠️  NumPy {_np.__version__} — reinstalando 1.26.4')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4',
            '--target', '/usr/local/lib/python3.12/dist-packages/',
            '--no-deps', '-q'
        ])
        print('🔴 ACCIÓN: Runtime → Restart session y vuelve a ejecutar')
    else:
        print(f'✅ NumPy {_np.__version__} — compatible')
else:
    print('ℹ️  NumPy no instalado aún — continúa')

## PASO 1 — Instalar dependencias

In [ ]:
!pip install -q reportlab matplotlib psycopg2-binary
!apt-get install -q -y fonts-dejavu > /dev/null 2>&1
print('✅ Dependencias instaladas')

## PASO 2 — Configuración

In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR   = Path('/content/drive/MyDrive/auditoria-digi')
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTADO_PATH = OUTPUT_DIR / 'resultado_fase3.json'
PDF_PATH       = OUTPUT_DIR / 'informe_auditoria.pdf'

if RESULTADO_PATH.exists():
    print(f'✅ resultado_fase3.json encontrado ({RESULTADO_PATH.stat().st_size:,} bytes)')
    print(f'   PDF salida → {PDF_PATH}')
else:
    raise FileNotFoundError(f'resultado_fase3.json no encontrado en {OUTPUT_DIR}')

PG_CONFIG = {
    'host': 'localhost', 'port': 5432,
    'database': 'auditoria_digi',
    'user': 'auditor', 'password': 'CAMBIAR_EN_PRODUCCION',
}
POSTGRES_ACTIVO = False
print(f'   PostgreSQL: {"ACTIVADO" if POSTGRES_ACTIVO else "DESACTIVADO (modo test)"}')

## PASO 3 — Cargar resultado_fase3.json

In [ ]:
with open(RESULTADO_PATH, 'r', encoding='utf-8') as f:
    resultado = json.load(f)

print('✅ JSON cargado')
print(f'   Asesor:     {resultado["asesor"]}')
print(f'   Nota final: {resultado["nota_final"]:.2f} / 10')
st = resultado['estadisticas']
print(f'   Criterios:  {st["n_si"]} Sí · {st["n_no"]} No · {st["n_na"]} N/A')

## NODO 12a — Estilos y paleta de colores DIGI

> Paleta oficial DIGI: azul #003087, naranja #FF6600. Fuentes DejaVu para soporte UTF-8 completo.

In [ ]:
import io
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.platypus import (
    BaseDocTemplate, PageTemplate, Frame,
    Table, TableStyle, Paragraph, Spacer, PageBreak,
    Image as RLImage, HRFlowable, KeepTogether,
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas as rl_canvas

# Fuentes DejaVu (soporte ñ, á, é, etc.)
try:
    pdfmetrics.registerFont(TTFont('DJ',  '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'))
    pdfmetrics.registerFont(TTFont('DJB', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))
    pdfmetrics.registerFont(TTFont('DJI', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Oblique.ttf'))
    FN, FB, FI = 'DJ', 'DJB', 'DJI'
    print('✅ Fuentes DejaVu registradas')
except Exception as e:
    FN, FB, FI = 'Helvetica', 'Helvetica-Bold', 'Helvetica-Oblique'
    print(f'⚠️  Usando Helvetica: {e}')

# Paleta DIGI
C_BLUE   = colors.HexColor('#003087')
C_BLUE_L = colors.HexColor('#E8EEF7')
C_ORANGE = colors.HexColor('#FF6600')
C_SI     = colors.HexColor('#27AE60')
C_NO     = colors.HexColor('#C0392B')
C_NA     = colors.HexColor('#7F8C8D')
C_WHITE  = colors.white
C_LGRAY  = colors.HexColor('#ECF0F1')
C_DGRAY  = colors.HexColor('#2C3E50')
C_BORDER = colors.HexColor('#BDC3C7')

W, H = A4

def S(name, **kw):
    d = dict(fontName=FN, fontSize=9, leading=12, textColor=C_DGRAY)
    d.update(kw); return ParagraphStyle(name, **d)

S_HDR  = S('hdr', fontName=FB, fontSize=8,   textColor=C_WHITE, alignment=TA_CENTER)
S_SEC  = S('sec', fontName=FB, fontSize=10,  textColor=C_WHITE)
S_SUB  = S('sub', fontName=FB, fontSize=9,   textColor=C_BLUE)
S_CEL  = S('cel', fontSize=7.5, leading=9.5)
S_BLD  = S('bld', fontName=FB, fontSize=8,   leading=10)
S_NRM  = S('nrm', fontSize=8, leading=10)
S_FBK  = S('fbk', fontSize=9, leading=13)
S_FTR  = S('ftr', fontSize=7, textColor=C_NA, alignment=TA_CENTER)

def nota_color(n):
    if n >= 7.5: return C_SI
    if n >= 5.0: return colors.HexColor('#E67E22')
    return C_NO

print(f'✅ Estilos DIGI listos | Página A4: {W:.0f}×{H:.0f} pt')

## NODO 12b — Gráficas (matplotlib → PNG embebido)

> 3 gráficas: donut de distribución · barras por sección · gauge de nota final.

In [ ]:
def chart_donut(n_si, n_no, n_na):
    fig, ax = plt.subplots(figsize=(4, 3.8), facecolor='none')
    pairs = [(n_si, '#27AE60', f'Si ({n_si})'),
             (n_no, '#C0392B', f'No ({n_no})'),
             (n_na, '#7F8C8D', f'N/A ({n_na})')]
    vals  = [v for v,c,l in pairs if v > 0]
    clrs  = [c for v,c,l in pairs if v > 0]
    lbls  = [l for v,c,l in pairs if v > 0]
    wedges, _, auto = ax.pie(
        vals, colors=clrs, autopct='%1.0f%%', startangle=90,
        pctdistance=0.72,
        wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2))
    for at in auto:
        at.set_fontsize(9); at.set_color('white'); at.set_fontweight('bold')
    ax.legend(wedges, lbls, loc='lower center', bbox_to_anchor=(0.5,-0.12),
              ncol=3, fontsize=8, frameon=False)
    ax.set_title('Distribucion de respuestas', fontsize=9, fontweight='bold', pad=8)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', transparent=True)
    plt.close(fig); buf.seek(0); return buf

def chart_barras(nota_e, nota_a):
    fig, ax = plt.subplots(figsize=(5.5, 2.2), facecolor='none')
    secs  = ['Estructura\nde la llamada', 'Actitud /\nComunicacion']
    notas = [nota_e, nota_a]
    clrs  = ['#003087', '#FF6600']
    bars  = ax.barh(secs, notas, color=clrs, height=0.45, edgecolor='none')
    ax.set_xlim(0, 10)
    ax.axvline(5, color='#BDC3C7', lw=1, ls='--')
    for bar, n in zip(bars, notas):
        ax.text(min(bar.get_width()+0.15, 9.6), bar.get_y()+bar.get_height()/2,
                f'{n:.2f}', va='center', fontsize=11, fontweight='bold', color='#2C3E50')
    ax.set_xlabel('Puntuacion / 10', fontsize=8)
    ax.set_title('Puntuacion por seccion', fontsize=9, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', transparent=True)
    plt.close(fig); buf.seek(0); return buf

def chart_gauge(nota):
    fig, ax = plt.subplots(figsize=(3.8, 2.5), facecolor='none')
    for t_lo, t_hi, col in [(0,0.5,'#C0392B'),(0.5,0.75,'#E67E22'),(0.75,1,'#27AE60')]:
        t = np.linspace(np.pi*t_lo, np.pi*t_hi, 50)
        xo = np.cos(t); yo = np.sin(t)
        xi = np.cos(t[::-1])*0.62; yi = np.sin(t[::-1])*0.62
        ax.fill(np.concatenate([xo,xi]), np.concatenate([yo,yi]), color=col, alpha=0.85)
    angle = np.pi*(1-nota/10)
    ax.annotate('', xy=(np.cos(angle)*0.85, np.sin(angle)*0.85), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2.5, mutation_scale=14))
    ax.plot(0, 0, 'o', color='#2C3E50', ms=7, zorder=5)
    ax.text(0,-0.18, f'{nota:.2f}', ha='center', fontsize=20, fontweight='bold', color='#2C3E50')
    ax.text(0,-0.40, '/ 10', ha='center', fontsize=9, color='#7F8C8D')
    ax.set_xlim(-1.2,1.2); ax.set_ylim(-0.55,1.15)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title('Nota Final', fontsize=9, fontweight='bold', y=0.98)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', transparent=True)
    plt.close(fig); buf.seek(0); return buf

print('⏳ Generando graficas...')
st = resultado['estadisticas']
BUF_DONUT  = chart_donut(st['n_si'], st['n_no'], st['n_na'])
BUF_BARRAS = chart_barras(resultado['nota_estructura_llamada'],
                           resultado['nota_actitud_comunicacion'])
BUF_GAUGE  = chart_gauge(resultado['nota_final'])
print('✅ 3 graficas generadas (donut · barras · gauge)')

## NODO 12c — Funciones de construcción del PDF

> Canvas personalizado con cabecera/pie DIGI · Tabla de criterios con alternado · Portada · Resumen.

In [ ]:
# ─── Canvas con header/footer DIGI ──────────────────────────────
class DIGICanvas(rl_canvas.Canvas):
    def __init__(self, *args, **kwargs):
        self._res = kwargs.pop('resultado', {})
        self._pg_states = []
        super().__init__(*args, **kwargs)

    def showPage(self):
        self._pg_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        n = len(self._pg_states)
        for state in self._pg_states:
            self.__dict__.update(state)
            self._deco(n)
            rl_canvas.Canvas.showPage(self)
        rl_canvas.Canvas.save(self)

    def _deco(self, n_pages):
        self.saveState()
        pg = self._pageNumber
        # Top bar azul
        self.setFillColor(C_BLUE)
        self.rect(0, H-38, W, 38, fill=1, stroke=0)
        # Linea naranja
        self.setFillColor(C_ORANGE)
        self.rect(0, H-42, W, 4, fill=1, stroke=0)
        # Texto cabecera
        self.setFillColor(C_WHITE)
        self.setFont(FB, 11)
        self.drawString(1.5*cm, H-25, 'DIGI Spain Telecom')
        self.setFont(FN, 8)
        self.drawRightString(W-1.5*cm, H-25, 'Informe de Auditoria C2C — CONFIDENCIAL')
        # Footer gris
        self.setFillColor(C_LGRAY)
        self.rect(0, 0, W, 32, fill=1, stroke=0)
        self.setFillColor(C_DGRAY)
        self.setFont(FN, 7)
        asesor = self._res.get('asesor','')
        fecha  = self._res.get('fecha_auditoria','')
        self.drawString(1.5*cm, 11, f'Asesor: {asesor}  |  Auditoria: {fecha}')
        self.drawRightString(W-1.5*cm, 11, f'Pagina {pg} de {n_pages}')
        self.restoreState()


# ─── Tabla de criterios ───────────────────────────────────────────
def tabla_criterios(items):
    col_w = [0.9*cm, 9.2*cm, 1.3*cm, 1.3*cm, 4.8*cm]
    hdr = [
        Paragraph('<b>ID</b>', S_HDR),
        Paragraph('<b>Criterio de evaluacion</b>', S_HDR),
        Paragraph('<b>Resp.</b>', S_HDR),
        Paragraph('<b>Punt.</b>', S_HDR),
        Paragraph('<b>Comentario</b>', S_HDR),
    ]
    data = [hdr]
    ts = [
        ('BACKGROUND', (0,0),(-1,0), C_BLUE),
        ('LINEBELOW',  (0,0),(-1,0), 1.2, C_BLUE),
        ('GRID',       (0,0),(-1,-1), 0.3, C_BORDER),
        ('VALIGN',     (0,0),(-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0),(-1,-1), 4),
        ('BOTTOMPADDING',(0,0),(-1,-1), 4),
        ('LEFTPADDING', (0,0),(-1,-1), 4),
        ('RIGHTPADDING',(0,0),(-1,-1), 4),
    ]
    for i, item in enumerate(items):
        resp = item['respuesta']
        ts.append(('BACKGROUND',(0,i+1),(-1,i+1), C_BLUE_L if i%2==0 else C_WHITE))
        if resp == 'Si':
            rc, rt = C_SI, 'Si'
        elif resp == 'No':
            rc, rt = C_NO, 'No'
        else:
            rc, rt = C_NA, 'N/A'
        r_s = S(f'r{i}', fontName=FB, fontSize=7.5, textColor=rc, alignment=TA_CENTER)
        p_s = S(f'p{i}', fontSize=7.5, alignment=TA_CENTER)
        crit = ' *' if item['critico'] else ''
        punt_str = f"{item['puntuacion']:.2f}" if resp != 'N/A' else '-'
        data.append([
            Paragraph(f"<b>{item['id']}</b>{crit}", S_CEL),
            Paragraph(item['pregunta'], S_CEL),
            Paragraph(rt, r_s),
            Paragraph(punt_str, p_s),
            Paragraph(item.get('comentario') or '', S_CEL),
        ])
    t = Table(data, colWidths=col_w, repeatRows=1)
    t.setStyle(TableStyle(ts))
    return t


# ─── Portada ──────────────────────────────────────────────────────
def build_portada(res):
    story = [Spacer(1, 0.6*cm)]
    story.append(Paragraph('<b>INFORME DE AUDITORIA</b>',
        S('tit', fontName=FB, fontSize=22, textColor=C_BLUE, alignment=TA_CENTER)))
    story.append(Paragraph('Control de Calidad C2C — DIGI Spain Telecom',
        S('subt', fontSize=11, textColor=C_ORANGE, alignment=TA_CENTER)))
    story.append(Spacer(1, 0.4*cm))

    # Gauge
    BUF_GAUGE.seek(0)
    story.append(Table([[RLImage(BUF_GAUGE, width=6.5*cm, height=4.2*cm)]],
                       colWidths=[W-3*cm],
                       style=[('ALIGN',(0,0),(-1,-1),'CENTER')]))
    story.append(Spacer(1, 0.3*cm))

    # Ficha
    ficha = [
        [Paragraph('<b>Asesor</b>', S_BLD),         Paragraph(res['asesor'], S_NRM)],
        [Paragraph('<b>ID Grabacion</b>', S_BLD),    Paragraph(res['id_grabacion'], S_NRM)],
        [Paragraph('<b>Tipificacion</b>', S_BLD),    Paragraph(res['tipificacion'], S_NRM)],
        [Paragraph('<b>Fecha llamada</b>', S_BLD),   Paragraph(res['fecha_llamada'], S_NRM)],
        [Paragraph('<b>Fecha auditoria</b>', S_BLD), Paragraph(res['fecha_auditoria'], S_NRM)],
        [Paragraph('<b>Servicio</b>', S_BLD),        Paragraph(res['servicio'], S_NRM)],
    ]
    t_f = Table(ficha, colWidths=[3.8*cm, 8.5*cm])
    t_f.setStyle(TableStyle([
        ('BACKGROUND',(0,0),(0,-1), C_BLUE_L),
        ('FONTNAME',  (0,0),(0,-1), FB),
        ('GRID',      (0,0),(-1,-1), 0.3, C_BORDER),
        ('TOPPADDING',(0,0),(-1,-1), 5),
        ('BOTTOMPADDING',(0,0),(-1,-1), 5),
        ('LEFTPADDING',(0,0),(-1,-1), 6),
        ('VALIGN',    (0,0),(-1,-1), 'MIDDLE'),
    ]))
    story.append(t_f)
    story.append(Spacer(1, 0.4*cm))

    # Notas
    nf = res['nota_final']
    ne = res['nota_estructura_llamada']
    na = res['nota_actitud_comunicacion']
    t_n = Table([[
        Paragraph(f'<b>{nf:.2f}</b><br/>NOTA FINAL',
            S('nf', fontName=FB, fontSize=24, textColor=nota_color(nf), alignment=TA_CENTER, leading=28)),
        Paragraph(f'<b>{ne:.2f}</b><br/>Estructura llamada',
            S('ne', fontName=FB, fontSize=14, textColor=nota_color(ne), alignment=TA_CENTER, leading=18)),
        Paragraph(f'<b>{na:.2f}</b><br/>Actitud / Comunic.',
            S('na2', fontName=FB, fontSize=14, textColor=nota_color(na), alignment=TA_CENTER, leading=18)),
    ]], colWidths=[4.5*cm, 5.5*cm, 5.5*cm])
    t_n.setStyle(TableStyle([
        ('BACKGROUND',(0,0),(0,0), C_LGRAY),
        ('BACKGROUND',(1,0),(2,0), C_BLUE_L),
        ('BOX',       (0,0),(-1,-1), 1.0, C_BLUE),
        ('INNERGRID', (0,0),(-1,-1), 0.5, C_BLUE),
        ('TOPPADDING',(0,0),(-1,-1), 10),
        ('BOTTOMPADDING',(0,0),(-1,-1), 10),
        ('VALIGN',    (0,0),(-1,-1), 'MIDDLE'),
    ]))
    story.append(t_n)
    story.append(Spacer(1, 0.3*cm))

    # Estadisticas
    st = res['estadisticas']
    nc = st['n_criticos_fallados']
    alerta = f'<font color="#C0392B"><b>Atencion: {nc} critico(s) fallado(s)</b></font>  |  ' if nc else ''
    story.append(Paragraph(
        f'{alerta}Si: <b>{st["n_si"]}</b>  No: <b>{st["n_no"]}</b>  N/A: <b>{st["n_na"]}</b>',
        S('al', fontSize=8, alignment=TA_CENTER)))
    story.append(PageBreak())
    return story


# ─── Paginas de criterios ─────────────────────────────────────────
def build_criterios(res):
    story = []
    for sec_n, subsecs in res['secciones'].items():
        t_sec = Table([[Paragraph(f'  {sec_n}', S_SEC)]], colWidths=[W-3*cm])
        t_sec.setStyle(TableStyle([
            ('BACKGROUND',(0,0),(-1,-1), C_BLUE),
            ('TOPPADDING',(0,0),(-1,-1), 6),
            ('BOTTOMPADDING',(0,0),(-1,-1), 6),
            ('LEFTPADDING',(0,0),(-1,-1), 8),
        ]))
        story.append(t_sec)
        story.append(Spacer(1, 0.2*cm))
        for sub_n, items in subsecs.items():
            story.append(Paragraph(f'  {sub_n}', S_SUB))
            story.append(Spacer(1, 0.1*cm))
            story.append(KeepTogether([tabla_criterios(items)]))
            story.append(Spacer(1, 0.4*cm))
        story.append(Spacer(1, 0.2*cm))
    story.append(PageBreak())
    return story


# ─── Pagina de resumen ────────────────────────────────────────────
def build_resumen(res):
    story = []
    story.append(Paragraph('<b>Resumen de Rendimiento</b>',
        S('res', fontName=FB, fontSize=13, textColor=C_BLUE)))
    story.append(HRFlowable(width='100%', thickness=2, color=C_ORANGE, spaceAfter=10))

    BUF_DONUT.seek(0); BUF_BARRAS.seek(0)
    t_ch = Table([[
        RLImage(BUF_DONUT,  width=6.5*cm, height=5.2*cm),
        RLImage(BUF_BARRAS, width=8.5*cm, height=4.5*cm),
    ]], colWidths=[7*cm, 9.5*cm])
    t_ch.setStyle(TableStyle([
        ('VALIGN',(0,0),(-1,-1),'MIDDLE'),
        ('ALIGN', (0,0),(-1,-1),'CENTER'),
    ]))
    story.append(t_ch)
    story.append(Spacer(1, 0.5*cm))

    story.append(Paragraph('<b>Observaciones</b>',
        S('oh', fontName=FB, fontSize=10, textColor=C_BLUE)))
    story.append(Spacer(1, 0.1*cm))
    t_obs = Table([[Paragraph(res.get('observaciones',''), S_FBK)]], colWidths=[W-3*cm])
    t_obs.setStyle(TableStyle([
        ('BACKGROUND',(0,0),(-1,-1), C_BLUE_L),
        ('BOX',       (0,0),(-1,-1), 0.5, C_BORDER),
        ('TOPPADDING',(0,0),(-1,-1), 8),
        ('BOTTOMPADDING',(0,0),(-1,-1), 8),
        ('LEFTPADDING',(0,0),(-1,-1), 8),
        ('RIGHTPADDING',(0,0),(-1,-1), 8),
    ]))
    story.append(t_obs)
    story.append(Spacer(1, 0.4*cm))

    pf = res.get('puntos_fuertes', [])
    am = res.get('areas_mejora', [])
    half = (W-3*cm-0.3*cm)/2

    def fb_table(titulo, items, c_tit, c_tbg, c_rbg, c_box):
        rows = [[Paragraph(titulo, S('fth', fontName=FB, fontSize=9, textColor=c_tit))]]
        for item in items:
            rows.append([Paragraph(f'  {item}', S('fti', fontSize=8.5, leading=12))])
        t = Table(rows, colWidths=[half])
        t.setStyle(TableStyle([
            ('BACKGROUND',(0,0),(-1,0), c_tbg),
            ('BACKGROUND',(0,1),(-1,-1), c_rbg),
            ('BOX',       (0,0),(-1,-1), 0.5, c_box),
            ('TOPPADDING',(0,0),(-1,-1), 5),
            ('BOTTOMPADDING',(0,0),(-1,-1), 5),
            ('LEFTPADDING',(0,0),(-1,-1), 8),
        ]))
        return t

    t_pf = fb_table('Puntos Fuertes', pf,
        colors.HexColor('#1A6B3C'), colors.HexColor('#D5F5E3'),
        colors.HexColor('#EAFAF1'), C_SI)
    t_am = fb_table('Areas de Mejora', am,
        colors.HexColor('#922B21'), colors.HexColor('#FADBD8'),
        colors.HexColor('#FDEDEC'), C_NO)

    t_comb = Table([[t_pf, t_am]], colWidths=[half+0.15*cm, half+0.15*cm])
    t_comb.setStyle(TableStyle([
        ('VALIGN',(0,0),(-1,-1),'TOP'),
        ('LEFTPADDING',(0,0),(-1,-1),0),
        ('RIGHTPADDING',(0,0),(-1,-1),0),
        ('TOPPADDING',(0,0),(-1,-1),0),
        ('BOTTOMPADDING',(0,0),(-1,-1),0),
    ]))
    story.append(t_comb)
    story.append(Spacer(1, 0.5*cm))
    story.append(HRFlowable(width='100%', thickness=0.5, color=C_LGRAY, spaceAfter=4))
    story.append(Paragraph(
        f'Modelo LLM: {res.get("modelo_llm","Qwen3-8B")}  |  Pipeline: auditoria-digi v1.0  |  {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        S_FTR))
    return story


print('✅ Funciones PDF definidas (DIGICanvas, tabla_criterios, portada, criterios, resumen)')

## NODO 12d — Generar y guardar el PDF

In [ ]:
# Ensamblar story
story = []
story += build_portada(resultado)
story += build_criterios(resultado)
story += build_resumen(resultado)

# Margenes
LM = RM = 1.5*cm
frame = Frame(LM, 32+10, W-LM-RM, H-42-4-32-20, id='main')

doc = BaseDocTemplate(
    str(PDF_PATH), pagesize=A4,
    leftMargin=LM, rightMargin=RM,
    topMargin=42+4+10, bottomMargin=32+10,
)
doc.addPageTemplates([PageTemplate(id='digi', frames=[frame])])

def make_canvas(filename, *args, **kwargs):
    kwargs['resultado'] = resultado
    return DIGICanvas(filename, *args, **kwargs)

print('⏳ Generando PDF...')
doc.build(story, canvasmaker=make_canvas)

size_kb = PDF_PATH.stat().st_size / 1024
print(f'✅ PDF generado: {PDF_PATH.name}')
print(f'   Tamaño: {size_kb:.1f} KB')
print()
print('🔗 Para descargar desde Colab:')
print('   from google.colab import files')
print(f'   files.download("{PDF_PATH}")')

## NODO 13 — Registro en PostgreSQL

In [ ]:
DDL = '''
CREATE TABLE IF NOT EXISTS auditorias (
    id               SERIAL PRIMARY KEY,
    id_grabacion     VARCHAR(100) UNIQUE NOT NULL,
    asesor           VARCHAR(200),
    fecha_llamada    DATE,
    fecha_auditoria  DATE,
    tipificacion     VARCHAR(100),
    servicio         VARCHAR(50),
    nota_final       NUMERIC(4,2),
    nota_estructura  NUMERIC(4,2),
    nota_actitud     NUMERIC(4,2),
    n_si             INTEGER,
    n_no             INTEGER,
    n_na             INTEGER,
    n_criticos_fail  INTEGER,
    pdf_path         TEXT,
    modelo_llm       VARCHAR(100),
    resultado_json   JSONB,
    creado_en        TIMESTAMP DEFAULT NOW()
);
'''

def insertar_resultado(res, pdf_path, pg_config):
    import psycopg2, json as _j
    from datetime import date

    conn = psycopg2.connect(**pg_config)
    cur  = conn.cursor()
    cur.execute(DDL)

    def pdate(s):
        try: return date.fromisoformat(s) if s else None
        except: return None

    cur.execute(
        '''INSERT INTO auditorias (
               id_grabacion, asesor, fecha_llamada, fecha_auditoria,
               tipificacion, servicio, nota_final, nota_estructura,
               nota_actitud, n_si, n_no, n_na, n_criticos_fail,
               pdf_path, modelo_llm, resultado_json
           ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
           ON CONFLICT (id_grabacion) DO UPDATE SET
               nota_final=EXCLUDED.nota_final,
               nota_estructura=EXCLUDED.nota_estructura,
               nota_actitud=EXCLUDED.nota_actitud,
               resultado_json=EXCLUDED.resultado_json,
               creado_en=NOW()''',
        (
            res['id_grabacion'], res['asesor'],
            pdate(res.get('fecha_llamada')), pdate(res.get('fecha_auditoria')),
            res.get('tipificacion'), res.get('servicio'),
            res['nota_final'], res['nota_estructura_llamada'], res['nota_actitud_comunicacion'],
            res['estadisticas']['n_si'], res['estadisticas']['n_no'], res['estadisticas']['n_na'],
            res['estadisticas']['n_criticos_fallados'],
            str(pdf_path), res.get('modelo_llm'),
            _j.dumps(res, ensure_ascii=False),
        )
    )
    conn.commit(); cur.close(); conn.close()
    print(f'✅ Registro insertado/actualizado: {res["id_grabacion"]}')

# Ejecutar
if POSTGRES_ACTIVO:
    print('⏳ Conectando a PostgreSQL...')
    try:
        insertar_resultado(resultado, PDF_PATH, PG_CONFIG)
    except Exception as e:
        print(f'❌ Error PostgreSQL: {e}')
        print('   Revisa PG_CONFIG y disponibilidad del servidor')
else:
    print('⏭️  PostgreSQL desactivado (POSTGRES_ACTIVO = False)')
    print('   Para activar: ajusta PG_CONFIG y pon POSTGRES_ACTIVO = True')

## ✅ Verificación final

In [ ]:
print('=' * 55)
print('📋 RESUMEN FASE 4')
print('=' * 55)
if PDF_PATH.exists():
    print(f'✅ PDF:  {PDF_PATH.name}  ({PDF_PATH.stat().st_size/1024:.1f} KB)')
else:
    print('❌ PDF NO generado')
print(f'✅ Fuente: resultado_fase3.json')
print(f'   Nota final:   {resultado["nota_final"]:.2f} / 10')
st = resultado['estadisticas']
print(f'   Criterios:    {st["n_si"]+st["n_no"]+st["n_na"]} ({st["n_si"]} Si · {st["n_no"]} No · {st["n_na"]} N/A)')
print(f'   PostgreSQL:   {"Activado" if POSTGRES_ACTIVO else "Desactivado"}')
print('=' * 55)
print()
print('🚀 Fase 4 completada — siguiente: Fase 5 Docker')